# Views: bronze

`bronze_gedcom_with_record` -- each bronze GEDCOM line with its owning record xref. Split out of
`01_ingest_gedcom` so ingest never recreates views.

## Why views live in their own notebooks
`CREATE OR REPLACE VIEW` drops and recreates the view, which makes the **executing identity its
owner**. A view owned by someone else that reads it (e.g. `gold_person_evidence_summary`) then
fails with `INSUFFICIENT_PERMISSIONS`. So view DDL is kept out of the data-refresh notebooks:
views are *definitions*, not data, and only need (re)creating when a definition changes.

**Run this notebook only when a view definition changes**, not as part of a routine refresh.
Run it as the account that owns the views (or follow it with
`ALTER VIEW <name> OWNER TO <owner>` for each view it recreates).

Depends on: `bronze_gedcom` (loaded by `01_ingest_gedcom`).


In [0]:
%sql
CREATE OR REPLACE VIEW genealogy.bronze_gedcom_with_record AS
SELECT
  *,
  last_value(
    xref_id,
    true
  ) OVER (
    ORDER BY line_no
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS record_xref
FROM genealogy.bronze_gedcom;


In [0]:
%sql
SELECT
  record_xref,
  COUNT(*) AS lines
FROM genealogy.bronze_gedcom_with_record
GROUP BY record_xref
ORDER BY lines DESC
LIMIT 10;
